# Walkthrough example - Tide harmonic analysis
```{hint}
Before using the `harmonic_tide` function, make sure you have the necessary data and dependencies installed. This example assumes you have a time series of sea level data from a Sea Level Center - University of Hawaii database.
```

# Import requiered modules

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

The function for harmonic tide decomposition can be loaded by:

In [2]:
from oceanicospy.analysis.harmonic_tides import tide_harmonic_decomposition

/scratchsan/medellin/.conda/envs/oceanicospy-dev-jtorresto/lib/python3.13/site-packages/numpy/lib/_format_impl.py:838: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  array = pickle.load(fp, **pickle_kwargs)


# Sea level data

Method from the retrieval package are used to download data from the Sea Level Center of the University of Hawaii for the location of San Andrés Island with the h737 buoy

To download sea level data from retrieval methods:

In [3]:
from oceanicospy.retrievals.download_UHSLC_data import UHSLCDownloader

/scratchsan/medellin/.conda/envs/oceanicospy-dev-jtorresto/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Sea level data for 2013 at San Andrés Island is downloaded

In [16]:
downloader=UHSLCDownloader('737', '.', '2013-01-01', '2013-12-31', -5)
filepath=downloader.download()
data=downloader.clean_data(filepath)
data.head()

Downloaded h737.csv to h737.csv.


,depth[m]
2013-01-01 00:00:00,2.420
2013-01-01 01:00:00,2.406
2013-01-01 02:00:00,2.373
2013-01-01 03:00:00,2.331
2013-01-01 04:00:00,2.283


# Using `tide_harmonic_decomposition` function

The function will return 2 pandas dataframes. 

The first one will return a dataframe for the time period with the input sea level serie, the fitted astronomic tide serie, and the residual tide serie, which is coming from the substraction of the astronomic tide from the original sea level.

The second dataframe show the characteristics of each harmonic component estimated. The dataframe indicates name of the harmonic component (ie. K1, M2), amplitude (in the units of the sea level input), confidence interval of the amplitude, phase, confidence interval of the phase, percentage of energy of the harmonic component, and Signal to Noise Ratio.

We will provide to the function:
1. The sea level list
2. The time list
3. The latitude of the sensor location as integer. In this example is 12.57.
4. If we want to remove take into account linear tendency of the series. True/False Boleean.
5. The method for harmonic tide component estimation*. 
6. The method for confidence interval estimation*.

*For more technical details on the methods for harmonic tide components and confidence interval estimation refers to UTide python library documentation.



In [25]:
tideSeries, tideCoef = tide_harmonic_decomposition(data['depth[m]'].values, data.index,
                                                      lat=12.57,trend=False, method='ols', conf_int='MC', verbose=False)

solve: matrix prep ... solution ... done.


In [26]:
tideSeries.head()

,time,sea_level,tide,residual
0,2013-01-01 00:00:00,2.420,2.518624,-0.098624
1,2013-01-01 01:00:00,2.406,2.505216,-0.099216
2,2013-01-01 02:00:00,2.373,2.479791,-0.106791
3,2013-01-01 03:00:00,2.331,2.442832,-0.111832
4,2013-01-01 04:00:00,2.283,2.396407,-0.113407


In [27]:
tideCoef.head()

,name,Amplitude,A_ci,Phase,P_ci,PE,SNR
0,K1,0.092744,0.000496,312.069409,0.406320,42.667466,134255.745751
1,M2,0.072129,0.000663,301.727062,0.482390,25.806997,45415.732932
2,O1,0.056678,0.000632,306.288844,0.511458,15.934838,30942.994464
3,SSA,0.033725,0.013670,153.847243,23.316079,5.641815,23.381148
4,P1,0.027905,0.000635,312.595822,1.130081,3.862741,7416.285380
